In [1]:
from langgraph_sdk import get_client
from dotenv import load_dotenv
import os

load_dotenv()

# ---- SETUP ----
# Replace with your deployed LangGraph API URL and API key
DEPLOYMENT_URL = os.getenv("DEPLOYMENT_URL")  # e.g. If running locally via langgraph dev command: "http://localhost:2024" or your cloud deployment URL
API_KEY = os.getenv("LANGSMITH_API_KEY")  # If using deployed graph on LangGraph Platform
GRAPH_ID = "deep_agent"  # Name of the graph, normally set in your langgraph.json file.

# 1. Connect to the LangGraph server
client = get_client(url=DEPLOYMENT_URL, api_key=API_KEY)
print("🔗 Connected to LangGraph server")

🔗 Connected to LangGraph server


In [2]:
# 2. Create a new assistant configuration
print("🤖 Creating a new assistant configuration...")

assistant = await client.assistants.create(
    graph_id = GRAPH_ID,
    config={
        "configurable": {
            "system_prompt": "You are a helpful AI assistant. Always reply in the voice of a pirate from the 1600s.",
            "model": "anthropic:claude-haiku-4-5",
            "selected_tools": ["get_todays_date", "advanced_research"],
            "name": "Pirate Assistant"
        }
    },
    name="Pirate Assistant",
)

print("✅ Assistant created successfully!")
print(f"   📍 Assistant ID: {assistant['assistant_id']}")
print(f"   📝 Name: {assistant['name']}")
print(f"   🔢 Version: {assistant['version']}")
print(f"   🧠 Model: {assistant['config']['configurable']['model']}")
print(f"   🛠️  Tools: {', '.join(assistant['config']['configurable']['selected_tools'])}")

🤖 Creating a new assistant configuration...
✅ Assistant created successfully!
   📍 Assistant ID: d338cdd2-8e2d-4bbb-9b38-398023679fe9
   📝 Name: Pirate Assistant
   🔢 Version: 1
   🧠 Model: anthropic:claude-haiku-4-5
   🛠️  Tools: get_todays_date, advanced_research


In [ ]:
# 2. Create a new assistant configuration
print("🤖 Creating a new assistant configuration...")

assistant = await client.assistants.create(
    graph_id = GRAPH_ID,
    config={
        "configurable": {
            "system_prompt": "You are a helpful AI assistant. Always reply in the voice of a cowboy from the Old West.",
            "model": "anthropic:claude-haiku-4-5",
            "selected_tools": ["get_todays_date", "advanced_research", "finance_research"],
            "name": "Cowboy Assistant"
        }
    },
    name="Cowboy Assistant"
)

print("✅ Assistant created successfully!")
print(f"   📍 Assistant ID: {assistant['assistant_id']}")
print(f"   📝 Name: {assistant['name']}")
print(f"   🔢 Version: {assistant['version']}")
print(f"   🧠 Model: {assistant['config']['configurable']['model']}")
print(f"   🛠️  Tools: {', '.join(assistant['config']['configurable']['selected_tools'])}")

🤖 Creating a new assistant configuration...
✅ Assistant created successfully!
   📍 Assistant ID: ddcb9df5-8271-44e7-9e62-f5d9e2cfae11
   📝 Name: Parrot Assistant
   🔢 Version: 1
   🧠 Model: anthropic:claude-haiku-4-5
   🛠️  Tools: get_todays_date, advanced_research, finance_research


In [ ]:
# Store AGENTS.md in the LangGraph persistent store for each assistant
# content-writer/AGENTS.md → Pirate Assistant
# mcp-docs-agent/AGENTS.md → Cowboy Assistant
#
# StoreBackend expects value = {"content": list[str], "created_at": str, "modified_at": str}

from datetime import datetime, timezone

now = datetime.now(timezone.utc).isoformat()

# Read AGENTS.md files from disk
with open("../src/assistants/content-writer/AGENTS.md") as f:
    content_writer_content = f.read()

with open("../src/assistants/mcp-docs-agent/AGENTS.md") as f:
    mcp_docs_content = f.read()

# Map assistant name → AGENTS.md content
agents_md_by_name = {
    "Pirate Assistant": content_writer_content,
    "Cowboy Assistant": mcp_docs_content,
}

all_assistants = await client.assistants.search(graph_id=GRAPH_ID)
print(f"📋 Found {len(all_assistants)} assistant(s) for graph '{GRAPH_ID}'\n")

for asst in all_assistants:
    name = asst["name"]
    if name not in agents_md_by_name:
        continue

    content = agents_md_by_name[name]
    store_value = {
        "content": content.split("\n"),
        "created_at": now,
        "modified_at": now,
    }

    namespace = [asst["assistant_id"]]
    await client.store.put_item(namespace, key="/AGENTS.md", value=store_value)

    item = await client.store.get_item(namespace, key="/AGENTS.md")
    print(f"✅ Stored AGENTS.md for '{name}'")
    print(f"   └─ Source    : src/assistants/{'content-writer' if name == 'Pirate Assistant' else 'mcp-docs-agent'}/AGENTS.md")
    print(f"   └─ Namespace : {namespace}")
    print(f"   └─ Key       : {item['key']}")
    print(f"   └─ Lines     : {len(item['value']['content'])}")
    print(f"   └─ Updated   : {item['updated_at']}\n")


📋 Found 3 assistant(s) for graph 'deep_agent'

✅ Stored AGENTS.md for 'Parrot Assistant'
   └─ Source    : src/assistants/mcp-docs-agent/AGENTS.md
   └─ Namespace : ['ddcb9df5-8271-44e7-9e62-f5d9e2cfae11']
   └─ Key       : /AGENTS.md
   └─ Lines     : 40
   └─ Updated   : 2026-04-14T00:43:37.533209+00:00

✅ Stored AGENTS.md for 'Pirate Assistant'
   └─ Source    : src/assistants/content-writer/AGENTS.md
   └─ Namespace : ['d338cdd2-8e2d-4bbb-9b38-398023679fe9']
   └─ Key       : /AGENTS.md
   └─ Lines     : 32
   └─ Updated   : 2026-04-14T00:43:37.539490+00:00



In [10]:
# Stream token-by-token using stream_mode="messages"
thread = await client.threads.create()
print(f"🧵 Thread created: {thread['thread_id']}\n")

input_data = {"messages": [{"role": "human", "content": "What tools do you have available?"}]}

print("💬 Assistant: ", end="", flush=True)

# Track last seen text per message ID — each partial event contains cumulative content, not deltas
last_seen: dict[str, str] = {}

async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant["assistant_id"],
    input=input_data,
    stream_mode="messages",
):
    if chunk.event == "messages/partial":
        for msg in chunk.data:
            if msg.get("type") == "ai":
                msg_id = msg.get("id", "")
                for block in msg.get("content", []):
                    if block.get("type") == "text":
                        full_text = block.get("text", "")
                        prev_text = last_seen.get(msg_id, "")
                        # Only print the newly arrived portion
                        if full_text.startswith(prev_text):
                            print(full_text[len(prev_text):], end="", flush=True)
                        last_seen[msg_id] = full_text

print()  # newline at end


🧵 Thread created: c39b1f43-2252-403a-93d0-ea7988f887ea

💬 Assistant: I have access to the following tools:

**File & Filesystem Tools:**
- `ls` — List files in a directory
- `read_file` — Read file contents (with pagination support for large files)
- `write_file` — Create new files
- `edit_file` — Edit existing files
- `glob` — Find files matching patterns
- `grep` — Search for text within files

**Research & Information Tools:**
- `advanced_research` — Perform in-depth web research on a query
- `get_todays_date` — Get the current date

**Task Management Tools:**
- `write_todos` — Create and manage structured task lists for complex objectives
- `task` — Launch subagents to handle isolated, complex tasks in parallel

**Subagent Type:**
- `general-purpose` — A capable agent with access to all tools, useful for complex multi-step tasks, research, and file operations

These tools let me help you with research, file management, task organization, and complex multi-step work. What would you 